In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CLEAN_PATH = PROJECT_ROOT / 'clean' / 'store_sales_clean.parquet'
REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(CLEAN_PATH)
total_units = df['sales'].sum()
print(f'Clean dataset loaded: {len(df):,} rows, {df.shape[1]} columns.')
print(f'Total units sold across all stores and families: {total_units:,.0f}')

Clean dataset loaded: 2,785,464 rows, 21 columns.
Total units sold across all stores and families: 1,073,644,952


In [2]:
# Q1: Top categories and % share of total units sold
cat_units = df.groupby('family')['sales'].sum().sort_values(ascending=False)
top3 = cat_units.head(3)
print('Top 3 Categories by Units Sold:')
for cat, val in top3.items():
    print(f'  {cat}: {val:,.0f} units ({val/total_units*100:.2f}%)')
print(f'Top 3 combined share of units: {top3.sum()/total_units*100:.2f}%')

plt.figure(figsize=(10, 6))
sns.barplot(x=cat_units.head(10).values / 1e6, y=cat_units.head(10).index, palette='viridis', hue=cat_units.head(10).index, legend=False)
plt.title('Q1: Top 10 Product Categories by Total Units Sold (Millions)', fontsize=14, pad=12)
plt.xlabel('Total Units Sold (Millions)', fontsize=12)
plt.ylabel('Category (Family)', fontsize=12)
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'q1_top_categories.png', dpi=150)
plt.show()

Top 3 Categories by Units Sold:
  GROCERY I: 343,462,735 units (31.99%)
  BEVERAGES: 216,954,486 units (20.21%)
  PRODUCE: 122,704,685 units (11.43%)
Top 3 combined share of units: 63.63%


In [3]:
# Q2: Weekday sales patterns, busiest and quietest day (% vs average)
df['weekday'] = df['date'].dt.day_name()
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_mean = df.groupby('weekday')['sales'].mean().reindex(weekday_order)
overall_mean = df['sales'].mean()
pct_vs_avg = (day_mean / overall_mean - 1) * 100

print('Day of Week Sales (Average Units & % vs Overall Average):')
for d, val in day_mean.items():
    print(f'  {d}: {val:.2f} units ({pct_vs_avg[d]:+.2f}% vs average)')

busiest_day = day_mean.idxmax()
quietest_day = day_mean.idxmin()
print(f'Busiest day: {busiest_day} ({pct_vs_avg[busiest_day]:+.2f}% vs average)')
print(f'Quietest day: {quietest_day} ({pct_vs_avg[quietest_day]:+.2f}% vs average)')

plt.figure(figsize=(9, 5))
colors = ['#4A90E2' if d not in ['Saturday', 'Sunday'] else '#E74C3C' for d in weekday_order]
sns.barplot(x=day_mean.index, y=day_mean.values, palette=colors, hue=day_mean.index, legend=False)
plt.title('Q2: Average Units Sold by Day of Week', fontsize=14, pad=12)
plt.xlabel('Day of Week', fontsize=12)
plt.ylabel('Average Units per Category-Day', fontsize=12)
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'q2_weekday.png', dpi=150)
plt.show()

Day of Week Sales (Average Units & % vs Overall Average):
  Monday: 373.96 units (-2.98% vs average)
  Tuesday: 346.53 units (-10.10% vs average)
  Wednesday: 357.99 units (-7.12% vs average)
  Thursday: 304.83 units (-20.92% vs average)
  Friday: 349.51 units (-9.32% vs average)
  Saturday: 467.62 units (+21.32% vs average)
  Sunday: 497.64 units (+29.11% vs average)
Busiest day: Sunday (+29.11% vs average)
Quietest day: Thursday (-20.92% vs average)


In [4]:
# Q3: Average units per day for each year 2013-2017
df['year'] = df['date'].dt.year
daily_totals = df.groupby(['year', 'date'])['sales'].sum().reset_index()
annual_daily_mean = daily_totals.groupby('year')['sales'].mean()

print('Average units sold per day for each year 2013-2017:')
for yr, val in annual_daily_mean.items():
    print(f'  {yr}: {val:,.1f} units/day')

plt.figure(figsize=(9, 5))
sns.barplot(x=annual_daily_mean.index.astype(str), y=annual_daily_mean.values / 1e3, color='#2ECC71')
plt.title('Q3: Average Units Sold per Day for Each Year (2013-2017)', fontsize=14, pad=12)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Average Daily Units (Thousands)', fontsize=12)
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'q3_trend.png', dpi=150)
plt.show()

Average units sold per day for each year 2013-2017:
  2013: 384,709.6 units/day
  2014: 573,902.0 units/day
  2015: 659,945.5 units/day
  2016: 788,673.6 units/day
  2017: 855,581.8 units/day


In [5]:
# Q4: Categories with highest promotion lift %
promo_stats = df.groupby(['family', df['onpromotion'] > 0])['sales'].mean().unstack()
promo_stats.columns = ['no_promo', 'on_promo']
promo_stats['lift_%'] = (promo_stats['on_promo'] / promo_stats['no_promo'] - 1) * 100
promo_stats = promo_stats.sort_values(by='lift_%', ascending=False)

top3_promo = promo_stats.head(3)
print('Top 3 Categories by Promotion Lift %:')
for cat, row in top3_promo.iterrows():
    print(f'  {cat}: {row["lift_%"]:.2f}% (on_promo: {row["on_promo"]:.2f} vs no_promo: {row["no_promo"]:.2f})')
print('Note: this is a link, not proof.')

plt.figure(figsize=(10, 5))
sns.barplot(x=promo_stats.head(10)['lift_%'].values, y=promo_stats.head(10).index, palette='Blues_r', hue=promo_stats.head(10).index, legend=False)
plt.title('Q4: Top 10 Categories by Promotion Lift %', fontsize=14, pad=12)
plt.xlabel('Promotion Lift %', fontsize=12)
plt.ylabel('Category (Family)', fontsize=12)
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'q4_promotions.png', dpi=150)
plt.show()

Top 3 Categories by Promotion Lift %:
  SCHOOL AND OFFICE SUPPLIES: 3929.61% (on_promo: 44.53 vs no_promo: 1.11)
  BABY CARE: 1305.81% (on_promo: 1.66 vs no_promo: 0.12)
  PET SUPPLIES: 317.95% (on_promo: 15.84 vs no_promo: 3.79)
Note: this is a link, not proof.


In [6]:
# Q5: Festival effect table, buildup %, and holiday lift %
store_dates = df[['date', 'store_nbr', 'is_holiday', 'is_closed']].drop_duplicates().sort_values(['store_nbr', 'date'])
store_dates['next_h1'] = store_dates.groupby('store_nbr')['is_holiday'].shift(-1).fillna(0)
store_dates['next_h2'] = store_dates.groupby('store_nbr')['is_holiday'].shift(-2).fillna(0)
store_dates['is_buildup'] = ((store_dates['next_h1'] == 1) | (store_dates['next_h2'] == 1)) & (store_dates['is_holiday'] == 0)
store_dates['is_normal'] = (store_dates['is_holiday'] == 0) & (~store_dates['is_buildup']) & (store_dates['is_closed'] == 0)

merged_fest = pd.merge(df[['date', 'store_nbr', 'family', 'sales']], store_dates[['date', 'store_nbr', 'is_holiday', 'is_buildup', 'is_normal']], on=['date', 'store_nbr'])

fest_df = merged_fest.groupby('family').apply(lambda g: pd.Series({
    'buildup_mean': g[g['is_buildup']]['sales'].mean(),
    'holiday_mean': g[g['is_holiday'] == 1]['sales'].mean(),
    'normal_mean': g[g['is_normal']]['sales'].mean(),
}), include_groups=False).reset_index()

fest_df['buildup_%'] = (fest_df['buildup_mean'] / fest_df['normal_mean'] - 1) * 100
fest_df['holiday_lift_%'] = (fest_df['holiday_mean'] / fest_df['normal_mean'] - 1) * 100
fest_df = fest_df.round({'buildup_mean': 2, 'holiday_mean': 2, 'normal_mean': 2, 'buildup_%': 2, 'holiday_lift_%': 2})

fest_df.to_csv(REPORTS_DIR / 'festival_effect.csv', index=False)
print('Saved festival_effect.csv successfully.')

top5_b = fest_df.sort_values('buildup_%', ascending=False).head(5)
bot3_b = fest_df.sort_values('buildup_%', ascending=False).tail(3)
top3_h = fest_df.sort_values('holiday_lift_%', ascending=False).head(3)

print('
Top 5 Categories by buildup_%:')
for _, r in top5_b.iterrows():
    print(f'  {r["family"]}: {r["buildup_%"]:+.2f}%')

print('
Bottom 3 Categories by buildup_%:')
for _, r in bot3_b.iterrows():
    print(f'  {r["family"]}: {r["buildup_%"]:+.2f}%')

print('
Top 3 Categories by holiday_lift_%:')
for _, r in top3_h.iterrows():
    print(f'  {r["family"]}: {r["holiday_lift_%"]:+.2f}%')

plt.figure(figsize=(12, 6))
plot_fest = fest_df.sort_values('holiday_lift_%', ascending=False)
x = np.arange(len(plot_fest))
width = 0.35
plt.bar(x - width/2, plot_fest['buildup_%'], width, label='Buildup %', color='#3498DB')
plt.bar(x + width/2, plot_fest['holiday_lift_%'], width, label='Holiday Lift %', color='#E74C3C')
plt.xticks(x, plot_fest['family'], rotation=90, fontsize=9)
plt.ylabel('Percentage Lift (%)', fontsize=12)
plt.title('Q5: Festival Buildup % and Holiday Lift % by Category', fontsize=14, pad=12)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'q5_festival_buildup.png', dpi=150)
plt.show()

Saved festival_effect.csv successfully.

Top 5 Categories by buildup_%:
  FROZEN FOODS: +92.86%
  LIQUOR,WINE,BEER: +73.75%
  LAWN AND GARDEN: +24.14%
  MAGAZINES: +24.06%
  HOME AND KITCHEN I: +18.73%

Bottom 3 Categories by buildup_%:
  SEAFOOD: -7.45%
  BREAD/BAKERY: -9.79%
  LADIESWEAR: -18.40%

Top 3 Categories by holiday_lift_%:
  FROZEN FOODS: +222.70%
  LIQUOR,WINE,BEER: +123.80%
  BOOKS: +76.29%


In [7]:
# Q6: Median jumpiness (CV = std / mean) for perishable vs long-life
open_df = df[df['is_closed'] == 0]
pair_stats = open_df.groupby(['store_nbr', 'family', 'is_perishable'])['sales'].agg(['mean', 'std']).reset_index()
pair_stats = pair_stats[pair_stats['mean'] > 0]
pair_stats['cv'] = pair_stats['std'] / pair_stats['mean']

med_cv_perish = pair_stats[pair_stats['is_perishable'] == 1]['cv'].median()
med_cv_longlife = pair_stats[pair_stats['is_perishable'] == 0]['cv'].median()

print('Jumpiness (Coefficient of Variation) Analysis:')
print(f'  Median CV for Perishable: {med_cv_perish:.2f}')
print(f'  Median CV for Long-Life: {med_cv_longlife:.2f}')
harder = 'Long-life' if med_cv_longlife > med_cv_perish else 'Perishable'
print(f'Harder to predict group: {harder} (higher CV reflects higher relative volatility and intermittency).')

plt.figure(figsize=(8, 5))
pair_stats['group'] = pair_stats['is_perishable'].map({1: 'Perishable', 0: 'Long-Life'})
sns.boxplot(x='group', y='cv', data=pair_stats, palette=['#E67E22', '#34495E'], hue='group', legend=False, showfliers=False)
plt.title('Q6: Jumpiness (Coefficient of Variation) Distribution: Perishable vs Long-Life', fontsize=14, pad=12)
plt.xlabel('Category Group', fontsize=12)
plt.ylabel('Jumpiness (CV = std / mean)', fontsize=12)
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'q6_perishable_vs_longlife.png', dpi=150)
plt.show()

Jumpiness (Coefficient of Variation) Analysis:
  Median CV for Perishable: 0.38
  Median CV for Long-Life: 1.07
Harder to predict group: Long-life (higher CV reflects higher relative volatility and intermittency).


In [8]:
# Q7: Payday effect percentage
payday_sales = open_df[open_df['is_payday'] == 1]['sales'].mean()
non_payday_sales = open_df[open_df['is_payday'] == 0]['sales'].mean()
payday_lift = (payday_sales / non_payday_sales - 1) * 100

print(f'Payday average units: {payday_sales:.2f}')
print(f'Non-payday average units: {non_payday_sales:.2f}')
print(f'Payday %: {payday_lift:+.2f}%')

Payday average units: 392.91
Non-payday average units: 388.88
Payday %: +1.04%


## Key Findings

1. Q1: The top 3 categories are GROCERY I (31.99%), BEVERAGES (20.21%), and PRODUCE (11.43%), together making up 63.63% of all units sold.
2. Q2: Sunday is the busiest day at +29.11% vs average, while Thursday is the quietest day at -20.92% vs average.
3. Q3: Average units per day were 384,709.6 in 2013, 573,902.0 in 2014, 659,945.5 in 2015, 788,673.6 in 2016, and 855,581.8 in 2017.
4. Q4: The 3 categories with the highest promotion lift are SCHOOL AND OFFICE SUPPLIES (3929.61%), BABY CARE (1305.81%), and PET SUPPLIES (317.95%), though this is a link, not proof.
5. Q5: From festival_effect.csv, the top 5 categories by buildup_% are FROZEN FOODS (+92.86%), LIQUOR,WINE,BEER (+73.75%), LAWN AND GARDEN (+24.14%), MAGAZINES (+24.06%), HOME AND KITCHEN I (+18.73%), the bottom 3 are SEAFOOD (-7.45%), BREAD/BAKERY (-9.79%), LADIESWEAR (-18.40%), and the top 3 by holiday_lift_% are FROZEN FOODS (+222.70%), LIQUOR,WINE,BEER (+123.80%), BOOKS (+76.29%).
6. Q6: The median jumpiness (CV) is 0.38 for perishable categories versus 1.07 for long-life categories, making long-life items harder to predict due to their higher demand volatility.
7. Q7: Paydays produce a +1.04% lift in average units sold compared to non-payday days.